# 04 Feature Engineering

Notebook ini membentuk fitur transaksi yang dipakai untuk deteksi impulsive spending. Semua proses dibuat display-only dan reusable agar bisa dijalankan dari hasil cleaning di memory atau dari CSV cleaned read-only.

Output utama di memory:
- `df_features`: dataframe transaksi dengan fitur baru.
- `feature_summary_df`: ringkasan fitur per dataset.
- `new_feature_columns`: daftar kolom fitur yang ditambahkan.


## 1. Import Library


In [ ]:
from pathlib import Path
import warnings
import os
import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import files


## 2. Konfigurasi Path dan Tampilan


In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

cleaned_separate_path = project_root / 'data' / 'interim' / 'cleaned_separate'

alt.data_transformers.disable_max_rows()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore')

print(f'Project root         : {project_root}')
print(f'Cleaned separate path: {cleaned_separate_path}')
print('Mode                 : display-only. Tidak ada file output yang disimpan.')


Project root         : /content
Cleaned separate path: /content/data/interim/cleaned_separate
Mode                 : display-only. Tidak ada file output yang disimpan.


## 3. Bobot Fingo Signal


In [ ]:
W_NIGHT = 0.35
W_CATEGORY = 0.30
W_AMOUNT = 0.25
W_WEEKEND = 0.10


## 4. Konfigurasi Kategori


In [ ]:
STANDARD_CATEGORIES = ['Makanan', 'Transportasi', 'Hiburan', 'Belanja', 'Pendidikan', 'Kesehatan', 'Tagihan', 'Lainnya']
HEDONIC_CATEGORIES = {'Hiburan', 'Belanja'}
NEUTRAL_CATEGORIES = {'Makanan', 'Transportasi', 'Lainnya'}
UTIL_CATEGORIES = {'Pendidikan', 'Kesehatan', 'Tagihan'}

DIRECT_CATEGORY_MAP = {
    'food': 'Makanan',
    'food_and_drink': 'Makanan',
    'food_drink': 'Makanan',
    'grocery': 'Makanan',
    'groceries': 'Makanan',
    'restaurant': 'Makanan',
    'snacks': 'Makanan',
    'makanan': 'Makanan',
    'transportation': 'Transportasi',
    'transport': 'Transportasi',
    'travel': 'Transportasi',
    'commute': 'Transportasi',
    'train': 'Transportasi',
    'bus': 'Transportasi',
    'transportasi': 'Transportasi',
    'entertainment': 'Hiburan',
    'subscription': 'Hiburan',
    'movies': 'Hiburan',
    'gaming': 'Hiburan',
    'culture': 'Hiburan',
    'festivals': 'Hiburan',
    'tourism': 'Hiburan',
    'social_life': 'Hiburan',
    'hiburan': 'Hiburan',
    'shopping': 'Belanja',
    'apparel': 'Belanja',
    'clothing': 'Belanja',
    'beauty': 'Belanja',
    'grooming': 'Belanja',
    'household': 'Belanja',
    'gift': 'Belanja',
    'belanja': 'Belanja',
    'education': 'Pendidikan',
    'self_development': 'Pendidikan',
    'pendidikan': 'Pendidikan',
    'health': 'Kesehatan',
    'health_and_fitness': 'Kesehatan',
    'fitness': 'Kesehatan',
    'medical': 'Kesehatan',
    'kesehatan': 'Kesehatan',
    'utilities': 'Tagihan',
    'rent': 'Tagihan',
    'bills': 'Tagihan',
    'mobile_service_provider': 'Tagihan',
    'water': 'Tagihan',
    'internet': 'Tagihan',
    'electricity': 'Tagihan',
    'phone': 'Tagihan',
    'tagihan': 'Tagihan',
    'other': 'Lainnya',
    'unknown': 'Lainnya',
    'lainnya': 'Lainnya',
}

KEYWORD_CATEGORY_RULES = [
    ('Makanan', ['makanan', 'minuman', 'food', 'snack', 'grocery', 'restaurant']),
    ('Transportasi', ['transport', 'commute', 'travel', 'train', 'bus', 'ojek']),
    ('Hiburan', ['entertainment', 'subscription', 'movie', 'gaming', 'mainan', 'festival', 'netflix', 'culture']),
    (
        'Belanja',
        [
            'shopping', 'fashion', 'pakaian', 'apparel', 'beauty', 'aksesoris', 'plastik', 'wadah', 'rak',
            'celengan', 'nampan', 'tray', 'baskom', 'mangkok', 'lunch_box', 'rantang', 'saringan',
            'pintu', 'perkakas', 'seal', 'baut', 'roof', 'household',
        ],
    ),
    ('Pendidikan', ['education', 'school', 'course', 'book', 'buku']),
    ('Kesehatan', ['health', 'fitness', 'medical', 'obat', 'olahraga']),
    ('Tagihan', ['utilities', 'rent', 'bill', 'mobile', 'water', 'internet', 'electricity', 'phone', 'listrik', 'pulsa']),
]


## 5. Category Helper


In [ ]:
def clean_key(value):
    text = str(value).strip().lower().replace('&', ' and ')
    text = ''.join(character if character.isalnum() else '_' for character in text)

    while '__' in text:
        text = text.replace('__', '_')

    return text.strip('_')


def map_category(value, domain=None):
    key = clean_key(value)

    if key in DIRECT_CATEGORY_MAP:
        return DIRECT_CATEGORY_MAP[key]

    for category, keywords in KEYWORD_CATEGORY_RULES:
        if any(keyword in key for keyword in keywords):
            return category

    return 'Belanja' if domain == 'ecommerce_sales' else 'Lainnya'


## 6. Load Cleaned Data Helper


In [ ]:
def load_cleaned_from_csv(folder):
    frames = []

    for path in sorted(folder.glob('*_cleaned.csv')):
        frame = pd.read_csv(path, low_memory=False)
        if 'timestamp' not in frame.columns and 'date' in frame.columns:
            frame['timestamp'] = frame['date']
        if 'dataset_id' not in frame.columns:
            frame['dataset_id'] = path.name.replace('_cleaned.csv', '')
        frames.append(frame)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def prepare_cleaned_input(frame):
    data = frame.copy()
    required_columns = ['timestamp', 'amount', 'category', 'dataset_id']
    missing_columns = [column for column in required_columns if column not in data.columns]

    if missing_columns:
        raise ValueError(f'Kolom cleaned wajib belum ada: {missing_columns}')

    data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
    data['amount'] = pd.to_numeric(data['amount'], errors='coerce')
    data = data.dropna(subset=required_columns).query('amount > 0').copy()
    data['category'] = data.apply(lambda row: map_category(row['category'], row.get('domain')), axis=1)
    data['source'] = data['source'] if 'source' in data.columns else data['dataset_id']
    return data.sort_values('timestamp').reset_index(drop=True)


def resolve_cleaned_input(cleaned_frame=None):
    if isinstance(cleaned_frame, pd.DataFrame) and not cleaned_frame.empty:
        return prepare_cleaned_input(cleaned_frame), 'memory_from_03'

    return prepare_cleaned_input(load_cleaned_from_csv(cleaned_separate_path)), 'read_only_csv_fallback'


def build_input_summary(frame):
    return (
        frame.groupby('dataset_id')
        .agg(rows=('amount', 'size'), total_amount=('amount', 'sum'), median_amount=('amount', 'median'))
        .reset_index()
    )


## 7. Load Cleaned Data


In [ ]:
df_cleaned = pd.read_csv(
    'merged_finance_ecommerce_household_clean.csv',
    low_memory=False
)

df_cleaned = df_cleaned.rename(columns={
    'tanggal_transaksi': 'timestamp',
    'amount_idr': 'amount',
    'kategori_clean': 'category',
    'sumber_dataset': 'dataset_id'
})

df_input, input_mode = resolve_cleaned_input(df_cleaned)

print(f'Input mode : {input_mode}')

display(build_input_summary(df_input))
display(df_input.head())

Input mode : memory_from_03


,dataset_id,rows,total_amount,median_amount
0,daily_household,2452,1.237548e+09,18300.000
1,ecommerce_sales,4545,2.531376e+08,23523.000
2,personal_finance,1500,1.961281e+06,1156.285


,dataset_id,id_transaksi,timestamp,tahun,bulan,tahun_bulan,tipe_transaksi,category,amount,metode_pembayaran,status_transaksi,sumber_file,source
0,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Transportasi,1830.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household
1,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Makanan,73200.0,Credit Card,Expense,daily_household_transactions_clean.csv,daily_household
2,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Transportasi,3660.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household
3,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Transportasi,10980.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household
4,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Lainnya,7320.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household


## 8. Feature Helpers


In [ ]:
def add_time_features(frame):
    data = frame.copy()
    data['hour'] = data['timestamp'].dt.hour
    data['day_of_week'] = data['timestamp'].dt.dayofweek
    data['day_name'] = data['timestamp'].dt.day_name()
    data['day_of_month'] = data['timestamp'].dt.day
    data['month'] = data['timestamp'].dt.to_period('M').astype(str)
    data['date_only'] = data['timestamp'].dt.date.astype(str)
    data['is_weekend'] = data['day_of_week'].isin([5, 6]).astype(int)
    data['week_period'] = data['timestamp'].dt.isocalendar().week.astype(int)
    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    data['is_night'] = data['hour'].between(20, 23).astype(int)
    data['is_late_night'] = data['hour'].between(0, 3).astype(int)
    data['night_score'] = (1 - (np.minimum(data['hour'], 24 - data['hour']) / 12)).clip(0, 1)
    data['time_segment'] = pd.cut(
        data['hour'],
        bins=[-1, 5, 10, 15, 19, 23],
        labels=['late_night', 'morning', 'midday', 'evening', 'night']
    ).astype(str)

    return data


def add_category_features(frame):
    data = frame.copy()
    score_map = {category: 0.0 for category in STANDARD_CATEGORIES}
    score_map.update({category: 1.0 for category in HEDONIC_CATEGORIES})
    score_map.update({category: 0.5 for category in NEUTRAL_CATEGORIES})

    data['category_score'] = data['category'].map(score_map).fillna(0)
    data['category_type'] = np.select(
        [
            data['category'].isin(HEDONIC_CATEGORIES),
            data['category'].isin(NEUTRAL_CATEGORIES),
            data['category'].isin(UTIL_CATEGORIES),
        ],
        ['hedonic', 'neutral', 'utilitarian'],
        default='other',
    )
    data['is_hedonic_category'] = data['category'].isin(HEDONIC_CATEGORIES).astype(int)
    return data


def robust_z_score(series):
    values = pd.to_numeric(series, errors='coerce')
    median = values.median()
    mad = (values - median).abs().median()

    if pd.isna(mad) or mad == 0:
        std = values.std(ddof=0)
        if pd.isna(std) or std == 0:
            return pd.Series(0.0, index=series.index)
        return (values - values.mean()) / (std + 1e-9)

    return 0.6745 * (values - median) / (mad + 1e-9)


def add_amount_features(frame):
    data = frame.copy()
    data['amount_log'] = np.log1p(data['amount'])
    data['amount_z'] = data.groupby('dataset_id', group_keys=False)['amount'].transform(robust_z_score).clip(-3, 3)
    data['amount_score'] = ((data['amount_z'] + 3) / 6).clip(0, 1)
    data['amount_percentile'] = data.groupby('dataset_id')['amount'].rank(pct=True)
    data['is_high_amount'] = (data['amount_percentile'] >= 0.90).astype(int)
    return data


def add_context_features(frame):
    data = frame.copy().sort_values('timestamp').reset_index(drop=True)
    city = data.get('city', pd.Series('unknown', index=data.index)).astype('string').fillna('unknown')
    province = data.get('province', pd.Series('unknown', index=data.index)).astype('string').fillna('unknown')
    user_id = data.get('user_id', pd.Series('', index=data.index)).astype('string').fillna('')
    fallback_user = data['dataset_id'].astype(str) + '_' + city + '_' + province

    data['user_proxy'] = user_id.mask(user_id.str.strip().isin(['', '<NA>', 'nan']), fallback_user)
    data['transactions_same_day'] = data.groupby(['user_proxy', 'date_only'])['amount'].transform('size')
    data['daily_amount_total'] = data.groupby(['user_proxy', 'date_only'])['amount'].transform('sum')
    data['share_of_daily_spend'] = (data['amount'] / (data['daily_amount_total'] + 1e-9)).clip(0, 1)
    data['amount_vs_weekly_avg'] = (
        data['amount'] / (data.groupby(['user_proxy', 'week_period'])['amount'].transform('mean') + 1e-9)
    ).clip(0, 5)
    return data

def add_budget_features(data):
    data = data.copy()
    data['budget_limit'] = (
        data.groupby('user_proxy')['amount']
        .transform('mean') * 25
    )
    data['spent_so_far'] = (
        data.groupby(['user_proxy', 'week_period'])['amount']
        .cumsum()
    )
    data['budget_remaining_ratio'] = (
        (data['budget_limit'] - data['spent_so_far']) /
        (data['budget_limit'] + 1e-9)
    ).clip(0, 1)
    return data

def add_fingo_signal(frame):
    data = frame.copy()
    data['weekend_score'] = data['is_weekend'].astype(float)
    data['fingo_impulse_signal'] = (
        W_NIGHT * data['night_score']
        + W_CATEGORY * data['category_score']
        + W_AMOUNT * data['amount_score']
        + W_WEEKEND * data['weekend_score']
    ).clip(0, 1)
    q1 = data['fingo_impulse_signal'].quantile(0.25)
    q3 = data['fingo_impulse_signal'].quantile(0.75)
    data['signal_band'] = pd.cut(
        data['fingo_impulse_signal'],
        bins=[-np.inf, q1, q3, np.inf],
        labels=['low', 'watch', 'high'],
        include_lowest=True
    ).astype('string')
    return data


## 9. Run Feature Pipeline Helper


In [ ]:
def run_feature_engineering(frame):
    data = add_time_features(frame)
    data = add_category_features(data)
    data = add_amount_features(data)
    data = add_context_features(data)
    data = add_budget_features(data)
    data = add_fingo_signal(data)
    return data


def get_new_feature_columns(input_frame, output_frame):
    base_columns = set(input_frame.columns)
    return [column for column in output_frame.columns if column not in base_columns]


## 10. Run Feature Pipeline


In [ ]:
df_features = run_feature_engineering(df_input)
new_feature_columns = get_new_feature_columns(df_input, df_features)

print(f'Input shape : {df_input.shape}')
print(f'Output shape: {df_features.shape}')
print(f'Fitur baru  : {len(new_feature_columns)} kolom')
display(pd.DataFrame({'new_feature_columns': new_feature_columns}))
display(df_features.head())


Input shape : (8497, 13)
Output shape: (8497, 46)
Fitur baru  : 33 kolom


,new_feature_columns
0,hour
1,day_of_week
2,day_name
3,day_of_month
4,month
5,date_only
6,is_weekend
7,week_period
8,hour_sin
9,hour_cos


,dataset_id,id_transaksi,timestamp,tahun,bulan,tahun_bulan,tipe_transaksi,category,amount,metode_pembayaran,status_transaksi,sumber_file,source,hour,day_of_week,day_name,day_of_month,month,date_only,is_weekend,week_period,hour_sin,hour_cos,is_night,is_late_night,night_score,time_segment,category_score,category_type,is_hedonic_category,amount_log,amount_z,amount_score,amount_percentile,is_high_amount,user_proxy,transactions_same_day,daily_amount_total,share_of_daily_spend,amount_vs_weekly_avg,budget_limit,spent_so_far,budget_remaining_ratio,weekend_score,fingo_impulse_signal,signal_band
0,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Transportasi,1830.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,7.512618,-0.714176,0.380971,0.037520,0,daily_household_unknown_unknown,11,174216.0,0.010504,0.002797,1.261774e+07,1830.0,0.999855,0.0,0.595243,watch
1,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Makanan,73200.0,Credit Card,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,11.200964,2.380588,0.896765,0.694535,0,daily_household_unknown_unknown,11,174216.0,0.420168,0.111884,1.261774e+07,75030.0,0.994054,0.0,0.724191,high
2,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Transportasi,3660.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,8.205492,-0.634824,0.394196,0.116232,0,daily_household_unknown_unknown,11,174216.0,0.021008,0.005594,1.261774e+07,78690.0,0.993764,0.0,0.598549,watch
3,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Transportasi,10980.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,9.303922,-0.317412,0.447098,0.379894,0,daily_household_unknown_unknown,11,174216.0,0.063025,0.016783,1.261774e+07,89670.0,0.992893,0.0,0.611775,watch
4,daily_household,NaN,2015-01-01,2015,1,2015-01,Expense,Lainnya,7320.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,8.898502,-0.476118,0.420647,0.296900,0,daily_household_unknown_unknown,11,174216.0,0.042017,0.011188,1.261774e+07,96990.0,0.992313,0.0,0.605162,watch


In [ ]:
#Menghapus kolom id_transaksi karena tidak digunakan
df_features = df_features.drop(columns=['id_transaksi'])

display(df_features.head())

,dataset_id,timestamp,tahun,bulan,tahun_bulan,tipe_transaksi,category,amount,metode_pembayaran,status_transaksi,sumber_file,source,hour,day_of_week,day_name,day_of_month,month,date_only,is_weekend,week_period,hour_sin,hour_cos,is_night,is_late_night,night_score,time_segment,category_score,category_type,is_hedonic_category,amount_log,amount_z,amount_score,amount_percentile,is_high_amount,user_proxy,transactions_same_day,daily_amount_total,share_of_daily_spend,amount_vs_weekly_avg,budget_limit,spent_so_far,budget_remaining_ratio,weekend_score,fingo_impulse_signal,signal_band
0,daily_household,2015-01-01,2015,1,2015-01,Expense,Transportasi,1830.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,7.512618,-0.714176,0.380971,0.037520,0,daily_household_unknown_unknown,11,174216.0,0.010504,0.002797,1.261774e+07,1830.0,0.999855,0.0,0.595243,watch
1,daily_household,2015-01-01,2015,1,2015-01,Expense,Makanan,73200.0,Credit Card,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,11.200964,2.380588,0.896765,0.694535,0,daily_household_unknown_unknown,11,174216.0,0.420168,0.111884,1.261774e+07,75030.0,0.994054,0.0,0.724191,high
2,daily_household,2015-01-01,2015,1,2015-01,Expense,Transportasi,3660.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,8.205492,-0.634824,0.394196,0.116232,0,daily_household_unknown_unknown,11,174216.0,0.021008,0.005594,1.261774e+07,78690.0,0.993764,0.0,0.598549,watch
3,daily_household,2015-01-01,2015,1,2015-01,Expense,Transportasi,10980.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,9.303922,-0.317412,0.447098,0.379894,0,daily_household_unknown_unknown,11,174216.0,0.063025,0.016783,1.261774e+07,89670.0,0.992893,0.0,0.611775,watch
4,daily_household,2015-01-01,2015,1,2015-01,Expense,Lainnya,7320.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,8.898502,-0.476118,0.420647,0.296900,0,daily_household_unknown_unknown,11,174216.0,0.042017,0.011188,1.261774e+07,96990.0,0.992313,0.0,0.605162,watch


## 11. Feature Summary Helper


In [ ]:
def build_feature_summary(frame):
    return (
        frame.groupby('dataset_id')
        .agg(
            rows=('amount', 'size'),
            avg_signal=('fingo_impulse_signal', 'mean'),
            avg_amount_score=('amount_score', 'mean'),
            night_share=('is_night', lambda value: value.mean() * 100),
            hedonic_share=('is_hedonic_category', lambda value: value.mean() * 100),
            high_signal_share=('signal_band', lambda value: (value == 'high').mean() * 100),
        )
        .reset_index()
    )


## 12. Feature Summary


In [ ]:
feature_summary_df = build_feature_summary(df_features)
display(feature_summary_df)


,dataset_id,rows,avg_signal,avg_amount_score,night_share,hedonic_share,high_signal_share
0,daily_household,2452,0.577266,0.642603,14.274062,0.000000,28.915171
1,ecommerce_sales,4545,0.580407,0.597297,12.849285,100.000000,17.051705
2,personal_finance,1500,0.673522,0.524353,0.000000,19.533333,42.533333


## 15. Output Feature Engineering


In [ ]:
df_features.to_csv('df_features.csv', index=False)
feature_summary_df.to_csv('feature_summary_df.csv', index=False)
files.download('df_features.csv')
files.download('feature_summary_df.csv')
print('File berhasil disimpan')
print('- df_features.csv')
print('- feature_summary_df.csv')
print('Data siap dipakai: df_features, feature_summary_df, new_feature_columns')
print(f"Lokasi df_features.csv           : {os.path.abspath('df_features.csv')}")
print(f"Lokasi feature_summary_df.csv   : {os.path.abspath('feature_summary_df.csv')}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File berhasil disimpan
- df_features.csv
- feature_summary_df.csv
Data siap dipakai: df_features, feature_summary_df, new_feature_columns
Lokasi df_features.csv           : /content/df_features.csv
Lokasi feature_summary_df.csv   : /content/feature_summary_df.csv


In [ ]:
df_features = pd.read_csv('df_features.csv')

display(df_features.head())

,dataset_id,timestamp,tahun,bulan,tahun_bulan,tipe_transaksi,category,amount,metode_pembayaran,status_transaksi,sumber_file,source,hour,day_of_week,day_name,day_of_month,month,date_only,is_weekend,week_period,hour_sin,hour_cos,is_night,is_late_night,night_score,time_segment,category_score,category_type,is_hedonic_category,amount_log,amount_z,amount_score,amount_percentile,is_high_amount,user_proxy,transactions_same_day,daily_amount_total,share_of_daily_spend,amount_vs_weekly_avg,budget_limit,spent_so_far,budget_remaining_ratio,weekend_score,fingo_impulse_signal,signal_band
0,daily_household,2015-01-01 00:00:00,2015,1,2015-01,Expense,Transportasi,1830.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,7.512618,-0.714176,0.380971,0.037520,0,daily_household_unknown_unknown,11,174216.0,0.010504,0.002797,1.261774e+07,1830.0,0.999855,0.0,0.595243,watch
1,daily_household,2015-01-01 00:00:00,2015,1,2015-01,Expense,Makanan,73200.0,Credit Card,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,11.200964,2.380588,0.896765,0.694535,0,daily_household_unknown_unknown,11,174216.0,0.420168,0.111884,1.261774e+07,75030.0,0.994054,0.0,0.724191,high
2,daily_household,2015-01-01 00:00:00,2015,1,2015-01,Expense,Transportasi,3660.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,8.205492,-0.634824,0.394196,0.116232,0,daily_household_unknown_unknown,11,174216.0,0.021008,0.005594,1.261774e+07,78690.0,0.993764,0.0,0.598549,watch
3,daily_household,2015-01-01 00:00:00,2015,1,2015-01,Expense,Transportasi,10980.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,9.303922,-0.317412,0.447098,0.379894,0,daily_household_unknown_unknown,11,174216.0,0.063025,0.016783,1.261774e+07,89670.0,0.992893,0.0,0.611775,watch
4,daily_household,2015-01-01 00:00:00,2015,1,2015-01,Expense,Lainnya,7320.0,Cash,Expense,daily_household_transactions_clean.csv,daily_household,0,3,Thursday,1,2015-01,2015-01-01,0,1,0.0,1.0,0,1,1.0,late_night,0.5,neutral,0,8.898502,-0.476118,0.420647,0.296900,0,daily_household_unknown_unknown,11,174216.0,0.042017,0.011188,1.261774e+07,96990.0,0.992313,0.0,0.605162,watch


## Area Analisis Mandiri
Gunakan cell kosong di bawah untuk eksplorasi fitur setelah semua cell utama dijalankan.

Function dan variabel yang bisa dipakai ulang:
- `load_cleaned_from_csv(folder)`: membaca semua file `*_cleaned.csv` dari folder cleaned.
- `prepare_cleaned_input(frame)`: memastikan kolom penting bersih, timestamp valid, amount numerik, dan kategori standar.
- `resolve_cleaned_input(df_cleaned)`: memilih sumber data dari memory notebook 03 atau fallback CSV.
- `run_feature_engineering(frame)`: menjalankan seluruh pipeline fitur waktu, kategori, amount, context, dan signal.
- `get_new_feature_columns(df_input, df_features)`: melihat kolom fitur baru yang ditambahkan.
- `build_feature_summary(df_features)`: membuat ringkasan fitur per dataset.
- `plot_feature_overview(feature_summary_df, df_features)`: membuat visualisasi interaktif hasil feature engineering.
- `df_features`: dataframe utama dengan fitur baru.
- `feature_summary_df`: ringkasan fitur per dataset.
